# 🎮 Atari DQN Agent — Kaggle GPU Training Notebook

**Instructions:**
1. Enable GPU: Settings → Accelerator → GPU T4 x2
2. Enable Internet: Settings → Internet → ON
3. Run all cells in order
4. Download `latest_model.pth` and `best_model.pth` from Output panel
5. Paste both files into your local `models/` directory

In [ ]:
# ============================================================
# CELL 1 — INSTALL + REGISTER ALE
# ============================================================
!pip install -q autorom[accept-rom-license]
!AutoROM --accept-license

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)  # KEY FIX — registers ALE namespace

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import cv2
import os
import random
import matplotlib.pyplot as plt
from collections import deque
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device       :', device)
print('Torch version:', torch.__version__)
print('Gym version  :', gym.__version__)
print('ALE version  :', ale_py.__version__)
print('ALE registered ✓')
print('All imports done ✓')

In [ ]:
# ============================================================
# CELL 2 — HYPERPARAMETERS
# ============================================================
CONFIG = {
    'game'              : 'ALE/Pong-v5',
    'frame_skip'        : 4,
    'frame_size'        : 84,
    'stack_frames'      : 4,
    'episodes'          : 1000,
    'max_steps'         : 10000,
    'batch_size'        : 32,
    'replay_buffer_size': 100000,
    'learning_rate'     : 0.0001,
    'gamma'             : 0.99,
    'epsilon_start'     : 1.0,
    'epsilon_min'       : 0.05,
    'epsilon_decay'     : 0.995,
    'target_update'     : 1000,
    'min_replay_size'   : 10000,
    'save_every'        : 100,
    'checkpoint_path'   : '/kaggle/working/models/',
    'early_stop_reward' : 18.0,
}
os.makedirs(CONFIG['checkpoint_path'], exist_ok=True)
print('Config loaded ✓')

In [ ]:
# ============================================================
# CELL 3 — FRAME PREPROCESSING
# ============================================================
def preprocess_frame(frame):
    """RGB (H,W,3) → grayscale (84,84) float32 in [0,1]"""
    gray    = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (84, 84), interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0


class FrameStack:
    """Stack k consecutive preprocessed frames → (k, 84, 84)"""
    def __init__(self, k=4):
        self.k      = k
        self.frames = deque(maxlen=k)

    def reset(self, frame):
        processed = preprocess_frame(frame)
        for _ in range(self.k):
            self.frames.append(processed)
        return np.array(self.frames, dtype=np.float32)

    def step(self, frame):
        self.frames.append(preprocess_frame(frame))
        return np.array(self.frames, dtype=np.float32)


print('Preprocessing ready ✓')

In [ ]:
# ============================================================
# CELL 4 — ATARI ENVIRONMENT WRAPPER
# ============================================================
class AtariEnv:
    def __init__(self, game, frame_skip=4):
        self.env        = gym.make(game, frameskip=1, render_mode=None)
        self.frame_skip = frame_skip
        self.frame_stack = FrameStack(k=4)
        self.n_actions  = self.env.action_space.n

    def reset(self):
        obs, _ = self.env.reset()
        return self.frame_stack.reset(obs)

    def step(self, action):
        total_reward = 0.0
        frames       = []
        for _ in range(self.frame_skip):
            obs, reward, terminated, truncated, _ = self.env.step(action)
            total_reward += reward
            frames.append(obs)
            if terminated or truncated:
                break
        # Max pool over last 2 frames to remove flickering
        max_frame    = np.maximum(frames[-1], frames[-2]) if len(frames) >= 2 else frames[-1]
        total_reward = float(np.clip(total_reward, -1.0, 1.0))  # reward clipping
        state        = self.frame_stack.step(max_frame)
        done         = terminated or truncated
        return state, total_reward, done

    def close(self):
        self.env.close()


# Quick test
env_test = AtariEnv(CONFIG['game'], CONFIG['frame_skip'])
state    = env_test.reset()
print(f'State shape : {state.shape}')   # (4, 84, 84)
print(f'N actions   : {env_test.n_actions}')
env_test.close()
print('Environment ready ✓')

In [ ]:
# ============================================================
# CELL 5 — Q-NETWORK (CNN)
# ============================================================
class QNetwork(nn.Module):
    def __init__(self, n_actions):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4),  # (4,84,84)→(32,20,20)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), # →(64,9,9)
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), # →(64,7,7)
            nn.ReLU(),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        return self.fc(self.conv(x))


# Verify shapes
net        = QNetwork(6).to(device)
test_input = torch.zeros(1, 4, 84, 84).to(device)
print(f'Network output shape: {net(test_input).shape}')  # (1, 6)
print('Q-Network ready ✓')

In [ ]:
# ============================================================
# CELL 6 — EXPERIENCE REPLAY BUFFER
# ============================================================
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch                             = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.array(states,      dtype=np.float32),
            np.array(actions,     dtype=np.int64),
            np.array(rewards,     dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones,       dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


print('Replay Buffer ready ✓')

In [ ]:
# ============================================================
# CELL 7 — DQN AGENT
# ============================================================
class DQNAgent:
    def __init__(self, n_actions):
        self.n_actions  = n_actions
        self.epsilon    = CONFIG['epsilon_start']
        self.steps_done = 0

        # Main + Target networks
        self.q_net      = QNetwork(n_actions).to(device)
        self.target_net = QNetwork(n_actions).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer  = optim.Adam(self.q_net.parameters(),
                                     lr=CONFIG['learning_rate'])
        self.loss_fn    = nn.SmoothL1Loss()  # Huber loss
        self.buffer     = ReplayBuffer(CONFIG['replay_buffer_size'])

    def select_action(self, state):
        """Epsilon-greedy action selection."""
        if random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.q_net(s_t).argmax().item()

    def select_action_greedy(self, state):
        """Pure greedy — for evaluation."""
        with torch.no_grad():
            s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
            return self.q_net(s_t).argmax().item()

    def decay_epsilon(self):
        self.epsilon = max(
            CONFIG['epsilon_min'],
            self.epsilon * CONFIG['epsilon_decay']
        )

    def train_step(self):
        """Sample from buffer and update Q-network."""
        if len(self.buffer) < CONFIG['min_replay_size']:
            return None

        s, a, r, ns, d = self.buffer.sample(CONFIG['batch_size'])
        s_t  = torch.FloatTensor(s).to(device)
        a_t  = torch.LongTensor(a).to(device)
        r_t  = torch.FloatTensor(r).to(device)
        ns_t = torch.FloatTensor(ns).to(device)
        d_t  = torch.FloatTensor(d).to(device)

        # Q(s,a) for taken actions
        curr_q = self.q_net(s_t).gather(1, a_t.unsqueeze(1)).squeeze(1)

        # Bellman target: r + γ * max Q_target(s', a')
        with torch.no_grad():
            next_q   = self.target_net(ns_t).max(1)[0]
            target_q = r_t + CONFIG['gamma'] * next_q * (1 - d_t)

        loss = self.loss_fn(curr_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_net.parameters(), 10.0)
        self.optimizer.step()

        self.steps_done += 1
        if self.steps_done % CONFIG['target_update'] == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

        return loss.item()

    def save(self, path):
        torch.save({
            'q_net'     : self.q_net.state_dict(),
            'target_net': self.target_net.state_dict(),
            'optimizer' : self.optimizer.state_dict(),
            'epsilon'   : self.epsilon,
            'steps'     : self.steps_done,
        }, path)

    def load(self, path):
        ckpt = torch.load(path, map_location=device)
        self.q_net.load_state_dict(ckpt['q_net'])
        self.target_net.load_state_dict(ckpt['target_net'])
        self.optimizer.load_state_dict(ckpt['optimizer'])
        self.epsilon    = ckpt['epsilon']
        self.steps_done = ckpt['steps']
        print(f'Loaded — Steps: {self.steps_done} | ε: {self.epsilon:.4f}')


print('DQN Agent ready ✓')

In [ ]:
# ============================================================
# CELL 8 — TRAINING LOOP
# ============================================================
env   = AtariEnv(CONFIG['game'], CONFIG['frame_skip'])
agent = DQNAgent(env.n_actions)

episode_rewards = []
episode_losses  = []
avg_rewards     = []
best_avg        = -float('inf')
start_time      = time.time()

print('=' * 60)
print('Starting Training...')
print('=' * 60)

for episode in range(1, CONFIG['episodes'] + 1):
    state        = env.reset()
    total_reward = 0.0
    losses       = []
    ep_start     = time.time()

    for step in range(CONFIG['max_steps']):
        action                   = agent.select_action(state)
        next_state, reward, done = env.step(action)
        agent.buffer.push(state, action, reward, next_state, done)
        state        = next_state
        total_reward += reward
        loss = agent.train_step()
        if loss is not None:
            losses.append(loss)
        if done:
            break

    agent.decay_epsilon()

    episode_rewards.append(total_reward)
    avg_loss   = np.mean(losses) if losses else 0.0
    avg_reward = np.mean(episode_rewards[-100:])
    avg_rewards.append(avg_reward)
    episode_losses.append(avg_loss)
    duration = time.time() - ep_start

    # Console log every 10 episodes
    if episode % 10 == 0:
        elapsed = (time.time() - start_time) / 60
        print(f'Ep {episode:4d} | Reward: {total_reward:6.1f} | '
              f'Avg(100): {avg_reward:6.2f} | ε: {agent.epsilon:.3f} | '
              f'Loss: {avg_loss:.4f} | {elapsed:.1f}min')

    # Save checkpoint every N episodes
    if episode % CONFIG['save_every'] == 0:
        path = f"{CONFIG['checkpoint_path']}model_ep{episode}.pth"
        agent.save(path)
        print(f'  → Checkpoint saved: {path}')

    # Save best model
    if avg_reward > best_avg and episode >= 50:
        best_avg = avg_reward
        agent.save(f"{CONFIG['checkpoint_path']}best_model.pth")
        print(f'  ★ New best avg reward: {best_avg:.2f}')

    # Always save latest
    agent.save(f"{CONFIG['checkpoint_path']}latest_model.pth")

    # Early stopping
    if avg_reward >= CONFIG['early_stop_reward'] and episode >= 100:
        print(f'\n🎉 Solved at episode {episode}! Avg reward: {avg_reward:.2f}')
        break

env.close()
total_min = (time.time() - start_time) / 60
print(f'\nTraining complete in {total_min:.1f} minutes')
print(f'Best average reward: {best_avg:.2f}')

In [ ]:
# ============================================================
# CELL 9 — PLOT TRAINING CURVES
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DQN Training — ALE/Pong-v5', fontsize=14, fontweight='bold')

# Reward
axes[0].plot(episode_rewards, alpha=0.3, color='blue', label='Episode Reward')
axes[0].plot(avg_rewards, color='red', linewidth=2, label='Avg (100 ep)')
axes[0].axhline(y=10, color='green', linestyle='--', label='Target (10)')
axes[0].set_title('Reward per Episode')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(episode_losses, color='orange', alpha=0.7)
axes[1].set_title('Huber Loss per Episode')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

# Epsilon
epsilons = [
    max(CONFIG['epsilon_min'],
        CONFIG['epsilon_start'] * CONFIG['epsilon_decay'] ** i)
    for i in range(len(episode_rewards))
]
axes[2].plot(epsilons, color='green')
axes[2].set_title('Epsilon Decay')
axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Epsilon')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('training_curves.png saved ✓')

In [ ]:
# ============================================================
# CELL 10 — VERIFY OUTPUT FILES
# ============================================================
print('Files saved in /kaggle/working/models/:')
print('-' * 45)
for f in sorted(os.listdir(CONFIG['checkpoint_path'])):
    size = os.path.getsize(f"{CONFIG['checkpoint_path']}{f}") / 1e6
    print(f'  {f:<30} {size:.1f} MB')

print('\n Download from Kaggle Output panel → right side:')
print('  ✓ latest_model.pth  → paste into models/')
print('  ✓ best_model.pth    → paste into models/')
print('  ✓ training_curves.png')